# 📱 S6E8 | Predicting Smartphone Addiction
## End-to-End Pipeline: EDA → Feature Engineering → Ensemble → Submit
> **Metrik:** ROC-AUC | **Model:** LightGBM + XGBoost + CatBoost | **CV:** Stratified 5-Fold

## 📦 Section 1 — Setup & Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

# ── Settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams['figure.facecolor'] = '#0f0f1a'
plt.rcParams['axes.facecolor']   = '#0f0f1a'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'

SEED     = 42
N_SPLITS = 5

print('✅ Libraries loaded!')
print(f'   LightGBM : {lgb.__version__}')
print(f'   XGBoost  : {xgb.__version__}')

## 🔍 Section 2 — Load Data & EDA

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s6e8/train.csv')
test  = pd.read_csv('/kaggle/input/playground-series-s6e8/test.csv')
sub   = pd.read_csv('/kaggle/input/playground-series-s6e8/sample_submission.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
print(f'\nTarget distribution:')
print(train['addicted_label'].value_counts())
print(f'\nBalance: {train["addicted_label"].mean():.2%} positif')

In [ ]:
# ── Tampilkan Sample Data & Info
print('📋 Sample Train Data:')
display(train.head())
print('\n📋 Data Info:')
train.info()

In [ ]:
# ── Missing Values
missing = pd.DataFrame({
    'Train Missing'   : train.isnull().sum(),
    'Train Missing %' : (train.isnull().sum() / len(train) * 100).round(2),
    'Test Missing'    : test.isnull().sum(),
    'Test Missing %'  : (test.isnull().sum()  / len(test)  * 100).round(2),
})
print('🔍 Missing Values:')
display(missing[missing['Train Missing'] > 0] if missing['Train Missing'].sum() > 0 else 'Tidak ada missing value!')

In [ ]:
# ── Statistik Deskriptif
display(train.describe())

In [ ]:
# ── Target Distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🎯 Target: addicted_label', fontsize=16, fontweight='bold', color='white')

counts = train['addicted_label'].value_counts()
colors = ['#6C63FF', '#FF6584']

axes[0].bar(counts.index.astype(str), counts.values, color=colors, width=0.5, edgecolor='white', lw=0.5)
axes[0].set_title('Count per Class', color='white')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 50, f'{val:,}\n({val/len(train)*100:.1f}%)',
                 ha='center', va='bottom', color='white', fontweight='bold')

axes[1].pie(counts.values, labels=['Not Addicted (0)', 'Addicted (1)'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'color': 'white', 'fontsize': 11})
axes[1].set_title('Class Balance', color='white')

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribusi Fitur Numerik per Target Class
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['id', 'addicted_label']]

ncols = 3
nrows = (len(num_cols) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in zip([0, 1], ['#6C63FF', '#FF6584']):
        data = train[train['addicted_label'] == label][col].dropna()
        axes[i].hist(data, bins=30, alpha=0.65, color=color, label=f'Class {label}', edgecolor='none')
    axes[i].set_title(col, color='white', fontsize=9)
    axes[i].legend(fontsize=8)
    axes[i].set_facecolor('#0f0f1a')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('📊 Numerical Features Distribution by Target', fontsize=15, fontweight='bold', color='white', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap
corr = train[num_cols + ['addicted_label']].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdPu',
            ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('🔥 Correlation Matrix', fontsize=15, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## ⚙️ Section 3 — Feature Engineering

In [ ]:
def feature_engineering(df):
    df = df.copy()
    
    # ────────────────────────────────────────────────────────────
    # ⚠️ SESUAIKAN FITUR DI BAWAH INI SETELAH MELIHAT EDA!
    # Lihat nama kolom di output train.info() Section 2.
    # Hapus atau modifikasi blok 'if' yang tidak sesuai.
    # ────────────────────────────────────────────────────────────
    
    # Contoh: Rasio penggunaan harian terhadap waktu tidur
    if 'daily_usage_hours' in df.columns and 'sleep_hours' in df.columns:
        df['usage_sleep_ratio'] = df['daily_usage_hours'] / (df['sleep_hours'] + 1e-6)
    
    # Contoh: Rasio screen time sosial
    if 'social_media_usage' in df.columns and 'daily_usage_hours' in df.columns:
        df['social_pct'] = df['social_media_usage'] / (df['daily_usage_hours'] + 1e-6)
    
    # Contoh: Interaksi fitur yang paling berkorelasi dengan target
    if 'num_apps_installed' in df.columns and 'daily_usage_hours' in df.columns:
        df['apps_per_hour'] = df['num_apps_installed'] / (df['daily_usage_hours'] + 1e-6)
    
    # Contoh: Kuadratkan fitur yang paling penting
    # df['feature_sq'] = df['feature_name'] ** 2
    
    return df

train = feature_engineering(train)
test  = feature_engineering(test)

print(f'✅ Feature Engineering selesai!')
print(f'   Train: {train.shape} | Test: {test.shape}')

## 🔧 Section 4 — Preprocessing

In [ ]:
TARGET = 'addicted_label'
ID_COL = 'id'

cat_cols = train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns: {cat_cols}')

# ── Label Encoding
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_dict[col] = le
    print(f'  Encoded: {col}')

# ── Definisi fitur
all_features = [c for c in train.columns if c not in [ID_COL, TARGET]]
X_train = train[all_features]
y_train = train[TARGET]
X_test  = test[all_features]

print(f'\n✅ Preprocessing selesai!')
print(f'   Total features : {len(all_features)}')
print(f'   X_train        : {X_train.shape}')
print(f'   X_test         : {X_test.shape}')

## 🚀 Section 5 — Cross-Validation + Model Training

In [ ]:
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

In [ ]:
# ════════════════════════════════════════════════════════════
# ⚡ 5A — LightGBM
# ════════════════════════════════════════════════════════════
lgb_params = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'boosting_type'    : 'gbdt',
    'learning_rate'    : 0.02,
    'num_leaves'       : 127,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'min_child_samples': 20,
    'lambda_l1'        : 0.1,
    'lambda_l2'        : 0.1,
    'verbose'          : -1,
    'random_state'     : SEED,
}

oof_lgb  = np.zeros(len(X_train))
pred_lgb = np.zeros(len(X_test))
lgb_scores = []

print('🚀 Training LightGBM...')
print('=' * 55)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    tr_data  = lgb.Dataset(X_tr,  label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(
        lgb_params, tr_data,
        valid_sets=[val_data],
        num_boost_round=2000,
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(500)
        ]
    )
    
    oof_lgb[val_idx] = model.predict(X_val)
    pred_lgb += model.predict(X_test) / N_SPLITS
    
    score = roc_auc_score(y_val, oof_lgb[val_idx])
    lgb_scores.append(score)
    print(f'  Fold {fold+1} | AUC: {score:.5f} | Best iter: {model.best_iteration}')

lgb_oof = roc_auc_score(y_train, oof_lgb)
print('=' * 55)
print(f'✅ LightGBM OOF AUC : {lgb_oof:.5f}')
print(f'   Std Folds        : {np.std(lgb_scores):.5f}')

In [ ]:
# ════════════════════════════════════════════════════════════
# ⚡ 5B — XGBoost
# ════════════════════════════════════════════════════════════
xgb_params = {
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'auc',
    'learning_rate'    : 0.02,
    'max_depth'        : 6,
    'min_child_weight' : 5,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'gamma'            : 0.1,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'n_estimators'     : 2000,
    'random_state'     : SEED,
    'tree_method'      : 'hist',
    'device'           : 'cuda',  # hapus baris ini jika tidak ada GPU
}

oof_xgb  = np.zeros(len(X_train))
pred_xgb = np.zeros(len(X_test))
xgb_scores = []

print('🚀 Training XGBoost...')
print('=' * 55)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              early_stopping_rounds=100,
              verbose=500)
    
    oof_xgb[val_idx] = model.predict_proba(X_val)[:, 1]
    pred_xgb += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    score = roc_auc_score(y_val, oof_xgb[val_idx])
    xgb_scores.append(score)
    print(f'  Fold {fold+1} | AUC: {score:.5f}')

xgb_oof = roc_auc_score(y_train, oof_xgb)
print('=' * 55)
print(f'✅ XGBoost OOF AUC : {xgb_oof:.5f}')
print(f'   Std Folds       : {np.std(xgb_scores):.5f}')

In [ ]:
# ════════════════════════════════════════════════════════════
# ⚡ 5C — CatBoost
# ════════════════════════════════════════════════════════════
cb_params = {
    'iterations'          : 2000,
    'learning_rate'       : 0.02,
    'depth'               : 6,
    'loss_function'       : 'Logloss',
    'eval_metric'         : 'AUC',
    'random_seed'         : SEED,
    'l2_leaf_reg'         : 3.0,
    'bagging_temperature' : 0.5,
    'od_type'             : 'Iter',
    'od_wait'             : 100,
    'verbose'             : False,
    # 'task_type'         : 'GPU',  # Aktifkan jika ada GPU
}

oof_cb  = np.zeros(len(X_train))
pred_cb = np.zeros(len(X_test))
cb_scores = []

print('🚀 Training CatBoost...')
print('=' * 55)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    tr_pool  = Pool(X_tr,  label=y_tr)
    val_pool = Pool(X_val, label=y_val)
    
    model = CatBoostClassifier(**cb_params)
    model.fit(tr_pool, eval_set=val_pool, use_best_model=True)
    
    oof_cb[val_idx] = model.predict_proba(X_val)[:, 1]
    pred_cb += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    score = roc_auc_score(y_val, oof_cb[val_idx])
    cb_scores.append(score)
    print(f'  Fold {fold+1} | AUC: {score:.5f}')

cb_oof = roc_auc_score(y_train, oof_cb)
print('=' * 55)
print(f'✅ CatBoost OOF AUC : {cb_oof:.5f}')
print(f'   Std Folds        : {np.std(cb_scores):.5f}')

## 🏆 Section 6 — Ensembling & Pilih Prediksi Terbaik

In [ ]:
print('\n' + '=' * 55)
print('📊 MODEL COMPARISON')
print('=' * 55)
print(f'  LightGBM  OOF AUC : {lgb_oof:.5f}')
print(f'  XGBoost   OOF AUC : {xgb_oof:.5f}')
print(f'  CatBoost  OOF AUC : {cb_oof:.5f}')
print('=' * 55)

# ── 1. Simple Average
pred_avg = (pred_lgb + pred_xgb + pred_cb) / 3
oof_avg  = (oof_lgb  + oof_xgb  + oof_cb)  / 3
avg_score = roc_auc_score(y_train, oof_avg)
print(f'  Simple Average    : {avg_score:.5f}')

# ── 2. Weighted Average (berdasarkan OOF AUC)
total = lgb_oof + xgb_oof + cb_oof
w = [lgb_oof/total, xgb_oof/total, cb_oof/total]
pred_weighted = w[0]*pred_lgb + w[1]*pred_xgb + w[2]*pred_cb
oof_weighted  = w[0]*oof_lgb  + w[1]*oof_xgb  + w[2]*oof_cb
weighted_score = roc_auc_score(y_train, oof_weighted)
print(f'  Weighted Average  : {weighted_score:.5f}  (w={[f"{wi:.3f}" for wi in w]})')

# ── 3. Rank Averaging
def rank_avg(*arrays):
    return np.mean([rankdata(a) / len(a) for a in arrays], axis=0)

pred_rank = rank_avg(pred_lgb, pred_xgb, pred_cb)
oof_rank  = rank_avg(oof_lgb,  oof_xgb,  oof_cb)
rank_score = roc_auc_score(y_train, oof_rank)
print(f'  Rank Average      : {rank_score:.5f}')
print('=' * 55)

# ── Pilih otomatis metode terbaik
scores = {
    'Simple Average'  : (avg_score,      pred_avg),
    'Weighted Average': (weighted_score,  pred_weighted),
    'Rank Average'    : (rank_score,      pred_rank),
}
best_name = max(scores, key=lambda k: scores[k][0])
best_score, final_pred = scores[best_name]

print(f'\n🏆 Metode Terpilih : {best_name}')
print(f'   Final OOF AUC  : {best_score:.5f}')

## 💾 Section 7 — Generate & Download Submission

In [ ]:
# ── Buat submission.csv
submission = pd.DataFrame({
    'id'            : test[ID_COL],
    'addicted_label': final_pred
})

submission.to_csv('submission.csv', index=False)

print('✅ submission.csv berhasil dibuat!')
display(submission.head(10))
print(f'\nStatistik prediksi:')
display(submission['addicted_label'].describe())

In [ ]:
# ── Visualisasi distribusi prediksi
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(submission['addicted_label'], bins=50, color='#6C63FF', edgecolor='none', alpha=0.85)
ax.axvline(x=0.5, color='#FF6584', linestyle='--', lw=2, label='Threshold 0.5')
ax.set_title(f'📊 Final Prediction Distribution ({best_name})', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Probability', color='white')
ax.set_ylabel('Count', color='white')
ax.legend(labelcolor='white', fontsize=10)
ax.set_facecolor('#0f0f1a')
plt.tight_layout()
plt.show()

print(f'\n🎉 SELESAI!')
print(f'   Expected LB AUC ≈ {best_score:.5f}')
print(f'   → Klik tab Output (kanan bawah) → Download submission.csv')